# ターゲティング① — ベースライン：リーク列（通話時間）の除去と線形の物差しをテストで測る

出典：第03章 02節／パス：`03_targeting/01_baseline.ipynb`／パッケージ：numpy・pandas・scikit-learn・scipy／実行時間の目安：約2分

このノート単体で実行できる。OpenML bank-marketing（銀行販促の実データ）の取得に失敗したときは `input/bank.csv`（合成代替）、さらに欠品のときは内蔵生成へ切り替わる。
データの分け方（学習・検証・テスト）は `02_improved.ipynb` と同じである。最終評価はテストセットで1回だけ行う。


In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RS = 0
BANK_URL = "https://openml.org/data/v1/download/1586218/bank-marketing.arff"
# OpenML版の列名 V1〜V16 を UCI Bank Marketing の列名に戻す（列の順はUCIと同じ）
COLS = ["age", "job", "marital", "education", "default", "balance", "housing", "loan",
        "contact", "day", "month", "duration", "campaign", "pdays", "previous", "poutcome"]
# 通話時間は通話が終わるまで決まらない。架電先を選ぶ時点には存在しない（リーク列）
LEAK = ["duration"]


def make_bank_data(n: int, seed: int = 3) -> pd.DataFrame:
    # 公開データと同じ列名・型の合成代替（取得失敗時のみ使う手順確認用）
    r = np.random.default_rng(seed)
    prev = (r.poisson(1.5, size=n) + 1) * (r.uniform(size=n) < 0.18)
    d = pd.DataFrame({
        "age": r.integers(18, 90, size=n).astype(float),
        "job": r.choice(["blue-collar", "management", "technician", "admin.", "services", "retired"], size=n),
        "marital": r.choice(["married", "single", "divorced"], size=n, p=[0.60, 0.28, 0.12]),
        "education": r.choice(["secondary", "tertiary", "primary", "unknown"], size=n, p=[0.51, 0.29, 0.16, 0.04]),
        "default": r.choice(["no", "yes"], size=n, p=[0.98, 0.02]),
        "balance": np.round(r.lognormal(6.0, 1.5, size=n) - 300.0),
        "housing": r.choice(["yes", "no"], size=n, p=[0.56, 0.44]),
        "loan": r.choice(["no", "yes"], size=n, p=[0.84, 0.16]),
        "contact": r.choice(["cellular", "unknown", "telephone"], size=n, p=[0.65, 0.29, 0.06]),
        "day": r.integers(1, 32, size=n).astype(float),
        "month": r.choice(["may", "jul", "aug", "jun", "nov", "apr"], size=n),
        "duration": np.round(r.exponential(250.0, size=n)),
        "campaign": (r.poisson(1.8, size=n) + 1).astype(float),
        "pdays": np.where(prev > 0, r.integers(1, 400, size=n), -1).astype(float),
        "previous": prev.astype(float),
        "poutcome": np.where(prev > 0, r.choice(["failure", "other", "success"], size=n, p=[0.6, 0.15, 0.25]), "unknown"),
    })
    logit = (-3.6 + 0.004 * d["duration"] - 0.15 * d["campaign"] + 1.8 * (d["poutcome"] == "success")
             + 0.5 * (d["housing"] == "no") + 0.4 * (d["contact"] == "cellular"))
    y = (r.uniform(size=n) < 1.0 / (1.0 + np.exp(-logit))).astype(int)
    d["Class"] = np.where(y == 1, 2, 1)
    return d


def load_bank() -> pd.DataFrame:
    import io
    import urllib.request
    from scipy.io import arff
    raw = urllib.request.urlopen(BANK_URL, timeout=60).read()
    data, _ = arff.loadarff(io.StringIO(raw.decode("utf-8")))
    return pd.DataFrame(data)


def split3(X, y):
    # テスト25%を先に切り出し、残りを学習75%・検証25%に分ける（①②のノートで共通）
    X_tmp, X_te, y_tmp, y_te = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=RS)
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=RS)
    return X_tr, X_va, X_te, y_tr, y_va, y_te


def make_pre(X):
    num = [c for c in X.columns if X[c].dtype != object]
    cat = [c for c in X.columns if X[c].dtype == object]
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])


try:
    raw = load_bank()
    for c in raw.columns:
        if raw[c].dtype == object:
            raw[c] = raw[c].str.decode("utf-8")
    print("公開データを使用: OpenML bank-marketing（銀行販促の実データ）")
except Exception:
    try:
        raw = pd.read_csv("input/bank.csv")
        print("input/bank.csv（合成代替）を使用")
    except FileNotFoundError:
        raw = make_bank_data(45211)
        print("合成データを生成（手順確認用）")
raw = raw.rename(columns=dict(zip([f"V{j}" for j in range(1, 17)], COLS)))
y = (raw["Class"].astype(str) == "2").astype(int)  # 購読あり
X_all = raw.drop(columns=["Class"])
X = X_all.drop(columns=LEAK)
num_cols = [c for c in X.columns if X[c].dtype != object]
cat_cols = [c for c in X.columns if X[c].dtype == object]
print(f"件数: {len(raw)}, 購読率: {y.mean():.3f} / 数値{len(num_cols)}列・カテゴリ{len(cat_cols)}列（{', '.join(LEAK)}を除く）")


公開データを使用: OpenML bank-marketing（銀行販促の実データ）
件数: 45211, 購読率: 0.117 / 数値6列・カテゴリ9列（durationを除く）


In [2]:
# リークの実演：通話時間（duration）を特徴量に混ぜるとどうなるか
# 学習・テストの分け方は以降と同じ。違いは duration の有無だけである
Xl_tr, _, Xl_te, yl_tr, _, yl_te = split3(X_all, y)
m_leak = make_pipeline(make_pre(X_all), LogisticRegression(max_iter=5000)).fit(Xl_tr, yl_tr)
p_leak = m_leak.predict_proba(Xl_te)[:, 1]
print(f"[duration入り] PR-AUC={average_precision_score(yl_te, p_leak):.4f} / "
      f"ROC-AUC={roc_auc_score(yl_te, p_leak):.4f} ← 通話の結果を見ている")
# 本番では使えないので外す。以降は duration を除いた X で進める。

[duration入り] PR-AUC=0.5539 / ROC-AUC=0.9079 ← 通話の結果を見ている


In [3]:
X_tr, X_va, X_te, y_tr, y_va, y_te = split3(X, y)
dummy = DummyClassifier(strategy="prior").fit(X_tr, y_tr)
p_dummy = dummy.predict_proba(X_te)[:, 1]
print(f"[ダミー] PR-AUC={average_precision_score(y_te, p_dummy):.4f} / ROC-AUC={roc_auc_score(y_te, p_dummy):.4f}")
lr = make_pipeline(make_pre(X), LogisticRegression(max_iter=5000)).fit(X_tr, y_tr)
p_lr = lr.predict_proba(X_te)[:, 1]
print(f"[ベースライン・線形] PR-AUC={average_precision_score(y_te, p_lr):.4f} / ROC-AUC={roc_auc_score(y_te, p_lr):.4f}")
print(f"学習: {len(X_tr)}件 / テスト: {len(X_te)}件（検証用 {len(X_va)}件はベースラインでは使わない）")

[ダミー] PR-AUC=0.1170 / ROC-AUC=0.5000


[ベースライン・線形] PR-AUC=0.4137 / ROC-AUC=0.7688
学習: 25431件 / テスト: 11303件（検証用 8477件はベースラインでは使わない）


In [4]:
# リフト：上位層に反応者が何倍集まるか（施策の当たり方を見る）
order = np.argsort(-p_lr)
ys = y_te.to_numpy()[order]
base = ys.mean()
print("上位層 / 件数 / 反応率 / リフト:")
for frac in [0.05, 0.10, 0.20, 0.30]:
    k = int(len(ys) * frac)
    rate = ys[:k].mean()
    print(f"{frac:.2f} / {k:4d} / {rate:.3f} / {rate / base:.2f}倍")
assert np.isfinite(p_lr).all()

上位層 / 件数 / 反応率 / リフト:
0.05 /  565 / 0.563 / 4.81倍
0.10 / 1130 / 0.487 / 4.16倍
0.20 / 2260 / 0.331 / 2.83倍
0.30 / 3390 / 0.256 / 2.19倍


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
_book_fig = Path("../../../books/python-machine-jitumu/assets/images")
FIG = str(_book_fig if _book_fig.is_dir() else Path("outputs"))
Path(FIG).mkdir(parents=True, exist_ok=True)
order = np.argsort(-p_lr)
ys = y_te.to_numpy()[order]
base = ys.mean()
fracs = [0.05, 0.10, 0.20, 0.30]
lifts = [ys[:int(len(ys) * f)].mean() / base for f in fracs]
plt.figure(); plt.bar([str(f) for f in fracs], lifts)
plt.xlabel("top fraction"); plt.ylabel("lift"); plt.title("lift chart (test)")
plt.savefig(f"{FIG}/code-result-j03-01.png", dpi=150, bbox_inches="tight"); plt.close()
print("saved code-result-j03-01.png")


saved code-result-j03-01.png


## 読み取りどころ

- 通話時間（duration）を入れるとAUCが跳ね上がること（通話の結果を見ている証拠）。
- duration を除いた後のPR-AUC・ROC-AUCが正直な実力であること。
- 上位10%のリフト（何倍集まるか）が施策の当たり方の物差しになること。
